<div style="font-size: 0.85em; line-height: 1.5;">

<h3>Fusion – Reciprocal Rank Fusion (RRF)</h3>

<p><strong>What is it?</strong><br>
A method to combine results from multiple retrievers into one ranked list using the formula: <code>score = 1 / (k + rank)</code>, where <code>k</code> is a constant (often 60). The final documents are sorted by the sum of scores from all retrievers.</p>

<p><strong>How it works</strong></p>
<ol>
  <li>Run several retrievers on the same query.</li>
  <li>For each retriever, record the rank of every returned document.</li>
  <li>Calculate RRF score for each document across all retrievers.</li>
  <li>Sort documents by total score (descending).</li>
  <li>Return the top documents for answer generation.</li>
</ol>

<p><strong>Why use it?</strong></p>
<ul>
  <li><strong>Improves recall</strong> – captures documents missed by single retrievers.</li>
  <li><strong>Simple and effective</strong> – no training required.</li>
  <li><strong>Works with diverse retrievers</strong> – combines different strengths.</li>
</ul>

<p><strong>Visualisation</strong></p>
<pre>
Retriever A → [doc1, doc3]
Retriever B → [doc2, doc3]
        │
        ▼
RRF combination:
  doc1: score from A
  doc2: score from B
  doc3: score from A + B
        │
        ▼
Final ranked list
</pre>

<p><strong>Implementation</strong><br>
We will write a simple Python function to compute RRF scores from multiple retrievers.</p>

</div>

Step 1: Imports

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader, TextLoader, CSVLoader

from dotenv import load_dotenv

Step 2: Load Documents, Create Chunks, and Base Retriever

** We will:**
* Load the same health/agriculture documents we used earlier.
* Split them into chunks and add metadata (source, doc_type, language).
* Create an in‑memory Chroma vector store.
* Create a base retriever that returns the top‑k chunks.

This base retriever will be the foundation for the other retrievers we’ll use in fusion.



In [2]:
# Load relevant documents 

files = [
    ('../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf', 'pdf'),
    ('../../04_data_ingestion_document_processing/data/crop_disease.pdf', 'pdf'),
    ('../../04_data_ingestion_document_processing/data/agriculture.html', 'html'),
    ('../../04_data_ingestion_document_processing/data/agriculture.txt', 'txt'),
]

all_docs = []

for path, file_type in files:
    if file_type == 'pdf':
        loader = PyPDFLoader(path)
    elif file_type == 'html':
        loader = BSHTMLLoader(path, open_encoding='utf-8', bs_kwargs={'features': 'html.parser'})
    elif file_type == 'txt':
        loader = TextLoader(path, encoding='utf-8')
    else:
        continue
    all_docs.extend(loader.load())
    
# Split into chunks and add metadata
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get('source', '')
    file_name = source.split('\\')[-1] if '\\' in source else source.split('/')[-1]
    chunk.metadata['file_name'] = file_name
    chunk.metadata['doc_type'] = (
        'pdf' if file_name.endswith('.pdf')
        else 'html' if file_name.endswith('.html')
        else 'txt'
    )
    chunk.metadata['language'] = 'English'
    chunk.metadata['chunk_id'] = f'{file_name}_{i+1:03d}'
    
print(f'Create {len(chunks)} chunks.')

# Create embeddings and vectorstore (in-memory)
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=None
)

# Create embeddings and vectorstore (in-memory)
base_retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

print('Base retriever ready.')

Create 185 chunks.


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Base retriever ready.


Create LLM and Additional Retrievers

In [3]:
# Create a shared LLM
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# --- MultiQueryRetriever ---
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

print('MultiQueryRetriever ready.')

# --- SelfQueryRetriever ---
metadata_field_info = [
    AttributeInfo(
        name='source',
        description='The file path of the document. Contains keywords like "nigeria_health" or "crop_disease" or "agriculture".',
        type='string',
    ),
    AttributeInfo(
        name='doc_type',
        description='The type of document: pdf, html, txt',
        type='string',
    ),
    AttributeInfo(
        name='language',
        description='Language of the document, e.g., English',
        type='string',
    ),
]

document_content_description = 'Documents about agriculture, public health, and diseases in Nigeria'

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=False
)

print('SelfQueryRetriever ready.')

# --- ContextualCompressionRetriever ---
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)
print('ContextualCompressionRetriever ready.')

MultiQueryRetriever ready.
SelfQueryRetriever ready.
ContextualCompressionRetriever ready.


Implement Reciprocal Rank Fusion

The RRF formula is simple. For each document in the ranked list from a retriever, we compute:

* **score = 1 / (k + rank)**

where:

**rank starts at 1 for the top result.**

**k is a constant (often 60) to reduce the impact of very high ranks.**

We sum these scores across all retrievers for each document. Then we sort documents by total score descending and return the top top_n unique documents.

In [4]:
def reciprocal_rank_fusion(retriever_results, k=60, top_n=5):
    '''
    Combine multiple ranked lists of documents using Reciprocal Rank Fusion.

    Args:
        retriever_results: A list of lists, where each inner list is a ranked
                           list of Document objects from one retriever.
        k: constant to control rank influence (default 60).
        top_n: number of top documents to return after fusion.

    Returns:
        List of Document objects sorted by fused scores (descending).
    '''
    
    # store cumulative RRF score for each unique document
    fused_scores = {}
    
    # keep the actual Document object for each unique content
    doc_map = {}
    
    # Iterate over each retriever's ranked list
    for ranked_list in retriever_results:
        
        for rank, doc in enumerate(ranked_list, start=1):
            
            # Use the page_content as the unique key
            key = doc.page_content.strip()
            
            # Compute RRF score for this rank
            score = 1 / (k + rank)
            
            # Sum the score
            if key in fused_scores:
                fused_scores[key] += score
            
            else:
                fused_scores[key] = score
                doc_map[key] = doc
                
    # Sort keys by total score descending
    sorted_keys = sorted(fused_scores, key=fused_scores.get, reverse=True)
    
    return [doc_map[key] for key in sorted_keys[:top_n]]

We will:

1. Define a question.

2. Get results from each retriever:

* base_retriever

* multi_query_retriever

* self_query_retriever

* compression_retriever

3. Collect these into a list of lists.

4. Pass that list to reciprocal_rank_fusion().

5. Display the final fused documents.

This will show which documents are selected when we combine different retrieval strategies.

In [5]:
# Define a test question
question = 'What are the major diseases affecting crops and livestock in Nigeria, and how are they managed?'

# Retrieve list of documents from each retriever 
base_results = base_retriever.invoke(question)
multi_results = multi_query_retriever.invoke(question)
self_query_results = self_query_retriever.invoke(question)
compression_results = compression_retriever.invoke(question)

# Combine all result lists into one list for fusion
all_results = [
    base_results,
    multi_results,
    self_query_results,
    compression_results
]

# Perform Reciprocal Rank Fusion

fused_docs = reciprocal_rank_fusion(all_results, k=60, top_n=5)

# Display the fused results
print(f'Fused top {len(fused_docs)} documents:\n')
for i, doc in enumerate(fused_docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Fused top 5 documents:

1. Agriculture in Nigeria: Crops, Livestock, and Disease Management



Agriculture in Nigeria
Last updated: August 2026


Introduction
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
------------------------------------------------------------
2. 3. Mastitis
   Affected animal: Dairy cattle and goats
   Symptoms: Swollen udder, abnormal milk, fever.
   Control: Good milking hygiene and antibiotic treatment under veterinary guidance.

Challenges Facing Agriculture in Nigeria
   Source: ../../04_data_ingestion_document_processing/data/agriculture.txt
------------------------------------------------------------
3. Major Crops

Cassava – a major staple crop.
Yam – widely grown in the middle belt.
Maize – important for food and livestock feed.
Rice – local production is growing.
Sorghum and Millet – common in the north.
Beans – key source of protein.



Common C
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html


In [6]:
question = 'What are the common crop diseases and their control methods?'

base_results = base_retriever.invoke(question)
multi_results = multi_query_retriever.invoke(question)
self_query_results = self_query_retriever.invoke(question)
compression_results = compression_retriever.invoke(question)

all_results = [
    base_results,
    multi_results,
    self_query_results,
    compression_results
]

fused_docs = reciprocal_rank_fusion(all_results, k=60, top_n=5)

print(f'Fused top {len(fused_docs)} documents:\n')
for i, doc in enumerate(fused_docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)

Fused top 5 documents:

1. Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infecte
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
------------------------------------------------------------
2. Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cuttings, plant resistant varieties, and remove infected 
   Source: ../../04_data_ingestion_document_processing/data/agriculture.txt
------------------------------------------------------------
3. Management: 
 Use of resistant varieties. 
 Practice crop rotation. Cut out 
infected parts if only few.
   Source: ../../04_data_ingestion_document_processing/data/crop_disease.pdf
------